In [60]:
using PEPSKit, TensorKit

### Model Parameters ###
L = 3 #Length/width of unit cell
n_0 = round(Int, ((L-1)/2))+1 #Index of the point at the center of the lattice (1-based indexing).
m2 = 1.0 #Bare mass (squared)
m0 = 1 #Basis frequency
l = 0.1 #phi^4 coupling strength
Dim = 5 #Truncated local Hilbert space dimension
d = 2 #Number of spatial dimensions
a = 1.0 #Lattice spacing

### iPEPS Dimensions ###
Dbond = 6
χ = 48

### phi4 Hamiltonian ###
include("phi4_Hamiltonian.jl")
H, φ, φ2, φ4, Π, Π2 = phi4_model(L, m2, m0, l, Dim, d, a)

using JLD2
peps_1 = load_object("Z:/Energy Correlator 2D/VacStates/PEPS,L=1,m2=$m2,l=$l,a=$a,dim=$Dim,D=$Dbond,chi=$χ.jld2")
env_1 = load_object("Z:/Energy Correlator 2D/VacStates/env,L=1,m2=$m2,l=$l,a=$a,dim=$Dim,D=$Dbond,chi=$χ.jld2")

CTMRGEnv{TensorMap{ComplexF64, ComplexSpace, 1, 1, Vector{ComplexF64}}, TensorMap{ComplexF64, ComplexSpace, 3, 1, Vector{ComplexF64}}}(TensorMap{ComplexF64, ComplexSpace, 1, 1, Vector{ComplexF64}}[TensorMap{ComplexF64, ComplexSpace, 1, 1, Vector{ComplexF64}}(ComplexF64[-0.5620419029460201 + 0.8188231137596014im, 0.00013946908645864878 - 0.0003084668774789097im, -0.0002138446239156097 + 2.102579041003626e-5im, -7.282914648280937e-5 + 0.00010610274462842204im, 0.0002848447743749571 + 9.293270843104443e-5im, 2.5620765169447292e-5 - 3.732617552331589e-5im, 2.866934501033588e-6 - 4.176756798083559e-6im, -3.7994175626811797e-7 + 4.77732973482411e-7im, -1.048405239697953e-6 - 5.695904390634309e-7im, -5.854314837585503e-7 + 8.528989596372855e-7im  …  -4.5309233871243674e-8 + 1.6982449244467983e-8im, 4.167626825568388e-7 - 2.3492034115148664e-7im, 8.561180189696962e-7 - 4.825838836297721e-7im, 6.702012462144526e-7 - 3.777867476423576e-7im, -2.931904527518295e-7 + 5.6256057002303426e-8im, -5.190

In [61]:
#Convert the 1x1 vacuum tensor to LxL
A1 = peps_1.A[1,1] #1x1 vacuum tensor
AL = fill(A1, (L,L)) #Vacuum tensor copied over an LxL unit cell
peps_L = InfinitePEPS(AL)

InfinitePEPS{TensorMap{ComplexF64, ComplexSpace, 1, 4, Vector{ComplexF64}}}(TensorMap{ComplexF64, ComplexSpace, 1, 4, Vector{ComplexF64}}[TensorMap{ComplexF64, ComplexSpace, 1, 4, Vector{ComplexF64}}(ComplexF64[-0.018083935066639453 + 0.005609519856327898im, 0.05061655529110304 - 0.013739236897609359im, -0.046918658091732623 - 0.02841991044229403im, -0.0007962172811910623 + 0.0002386595538110202im, 0.00589635751742219 - 0.003992707032780721im, -0.006974526285502522 + 0.025779088971580627im, -0.007634202345919835 + 0.007905248422890905im, 0.003539349531789647 + 0.006776352944336623im, -0.0013105640612276236 + 0.0013303513183791067im, 0.003984427528970086 - 0.005577445215039894im  …  0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im, 0.0 + 0.0im], ℂ^5 ← (ℂ^6 ⊗ ℂ^6 ⊗ (ℂ^6)' ⊗ (ℂ^6)')) TensorMap{ComplexF64, ComplexSpace, 1, 4, Vector{ComplexF64}}(ComplexF64[-0.018083935066639453 + 0.005609519856327898im, 0.05061655529110304 

In [62]:
#Convert the 1x1 environment to LxL
env_L = CTMRGEnv(randn, ComplexF64, peps_L, ℂ^24); #Generate structure of LxL environment

#Replace corner and edge tensors of LxL environment with the 1x1 corner and edge tensors
for r in 1:L, c in 1:L
    for dir in 1:4
        setcorner!(env_L, corner(env_1, dir, 1, 1), dir, r, c)
    end

    for dir in 1:4
        setedge!(env_L, edge(env_1, dir, 1, 1), dir, r, c)
    end
end

In [63]:
### Save PEPS and CTMRG environment ###
save_object("Z:/Energy Correlator 2D/VacStates/PEPS,L=$L,m2=$m2,l=$l,a=$a,dim=$Dim,D=$Dbond,chi=$χ.jld2", peps_L)
save_object("Z:/Energy Correlator 2D/VacStates/env,L=$L,m2=$m2,l=$l,a=$a,dim=$Dim,D=$Dbond,chi=$χ.jld2", env_L)